# Лабораторная работа 4
## Начальная фаза симплекс-метода

Реализация алгоритма начальной фазы симплекс-метода для проверки совместности задачи в канонической форме и построения базисного допустимого плана.


In [9]:
import numpy as np

import numpy as np

def simplex_main_phase(c, A, x_init, B_init, max_iter=1000):
    """
    Основная фаза симплекс-метода.
    
    Параметры:
    c : np.ndarray
        Вектор коэффициентов целевой функции (на максимум).
    A : np.ndarray
        Матрица ограничений-равенств (m x n).
    x_init : np.ndarray
        Начальный базисный допустимый план (размер n).
    B_init : list или np.ndarray
        Набор индексов базисных переменных (размер m). Ожидается 0-индексация.
    max_iter : int
        Максимальное количество итераций (для защиты от зацикливания).
        
    Возвращает:
    status : str
        Сообщение о результате ('optimal' или 'unbounded').
    x : np.ndarray или None
        Оптимальный план, если целевая функция ограничена.
    B : list
        Индексы базисных переменных оптимального плана.
    """
    x = np.array(x_init, dtype=float)
    B = list(B_init)
    
    for iteration in range(max_iter):
        # 1. Построить базисную матрицу A_B и найти обратную
        A_B = A[:, B]
        try:
            A_B_inv = np.linalg.inv(A_B)
        except np.linalg.LinAlgError:
            raise ValueError("Матрица A_B вырождена. B не является базисом.")
            
        # 2. Сформировать вектор c_B
        c_B = c[B]
        
        # 3. Найти вектор потенциалов u
        u = c_B @ A_B_inv
        
        # 4. Найти вектор оценок delta
        delta = c - u @ A
        
        # 5. Проверка оптимальности (delta <= 0)
        # Учитываем вычислительную погрешность
        eps = 1e-9
        if np.all(delta <= eps):
            return "optimal", x, B
            
        # 6. Найти индекс первой положительной компоненты (j0)
        j0 = -1
        for j in range(len(delta)):
            if delta[j] > eps:
                j0 = j
                break
                
        # 7. Вычислить вектор z
        z = A_B_inv @ A[:, j0]
        
        # 8-9. Построить вектор theta и найти минимум
        theta = np.full(len(B), np.inf)
        for i in range(len(B)):
            if z[i] > eps:
                theta[i] = x[B[i]] / z[i]
                
        theta0 = np.min(theta)
        
        # 10. Проверка на неограниченность
        if np.isinf(theta0):
            return "unbounded", None, B
            
        # 11. Найти индекс k, на котором достигается минимум.
        k = np.argmin(theta)
        
        # 12-13. Замена базиса и обновление плана
        j_star = B[k]
        
        # Обновляем компоненты
        x[j0] = theta0
        for i in range(len(B)):
            if i != k:
                x[B[i]] = x[B[i]] - theta0 * z[i]
        x[j_star] = 0
        
        B[k] = j0
        
    raise RuntimeError("Превышено максимальное число итераций!")



def initial_phase_simplex(c, A, b, max_iter=1000, eps=1e-9):

    A_work = np.array(A, dtype=float).copy()
    b_work = np.array(b, dtype=float).copy()
    c = np.array(c, dtype=float).copy()

    m, n = A_work.shape

    # Шаг 1: делаем b >= 0
    for i in range(m):
        if b_work[i] < 0:
            b_work[i] *= -1
            A_work[i, :] *= -1

    # Шаг 2: вспомогательная задача
    I_m = np.eye(m)
    A_tilde = np.hstack([A_work, I_m])
    c_tilde = np.hstack([np.zeros(n), -np.ones(m)])

    # Шаг 3: начальный базисный допустимый план
    x_tilde0 = np.zeros(n + m)
    x_tilde0[n:] = b_work
    B = list(range(n, n + m))

    # Шаг 4: решаем вспомогательную задачу основной фазой
    status, x_tilde_opt, B = simplex_main_phase(
        c_tilde, A_tilde, x_tilde0, B, max_iter=max_iter
    )

    if status != "optimal":
        return {
            "feasible": False,
            "message": "Вспомогательная задача не решена до оптимума.",
            "x": None,
            "B": None,
            "A_reduced": None,
            "b_reduced": None,
        }

    # Шаг 5: проверка искусственных переменных
    if np.any(x_tilde_opt[n:] > eps):
        return {
            "feasible": False,
            "message": "Исходная задача несовместна (допустимых планов нет).",
            "x": None,
            "B": None,
            "A_reduced": None,
            "b_reduced": None,
        }

    # Шаг 6: допустимый план исходной задачи
    x = x_tilde_opt[:n].copy()

    # Шаги 7-9: убираем искусственные индексы из B
    while any(j >= n for j in B):
        # Шаг 8: выбираем максимальный искусственный индекс в B
        artificial_positions = [idx for idx, val in enumerate(B) if val >= n]
        k = max(artificial_positions, key=lambda idx: B[idx])
        j_k = B[k]

        A_B = A_tilde[:, B]
        A_B_inv = np.linalg.inv(A_B)

        # Повторный шаг 7-8: ищем замену среди не базисных реальных переменных
        nonbasic_real = [j for j in range(n) if j not in B]
        replacement = None
        for j in nonbasic_real:
            l_j = A_B_inv @ A_tilde[:, j]
            if abs(l_j[k]) > eps:
                replacement = j
                break

        if replacement is not None:
            B[k] = replacement
            continue

        # Шаг 9: ограничение избыточно, удаляем соответствующую строку
        col = A_tilde[:, j_k]
        candidate_rows = np.where(np.abs(col) > eps)[0]
        if len(candidate_rows) == 0:
            row_to_delete = k
        else:
            row_to_delete = int(candidate_rows[0])

        A_work = np.delete(A_work, row_to_delete, axis=0)
        b_work = np.delete(b_work, row_to_delete, axis=0)
        A_tilde = np.delete(A_tilde, row_to_delete, axis=0)
        B.pop(k)

        if len(B) == 0:
            break

    return {
        "feasible": True,
        "message": "Исходная задача совместна.",
        "x": x,
        "B": B,
        "A_reduced": A_work,
        "b_reduced": b_work,
    }


In [10]:
# Тестовый пример из задания
# x1 -> max
# x1 + x2 + x3 = 0
# 2x1 + 2x2 + 2x3 = 0
# x >= 0

c = np.array([1.0, 0.0, 0.0])
A = np.array([
    [1.0, 1.0, 1.0],
    [2.0, 2.0, 2.0],
])
b = np.array([0.0, 0.0])

result = initial_phase_simplex(c, A, b)

print("Статус:", result["message"])

if result["feasible"]:
    x = result["x"]
    B = result["B"]
    A_red = result["A_reduced"]
    b_red = result["b_reduced"]

    print("Допустимый план x^T =", np.round(x, 6))
    print("Базис B (0-based) =", B)
    print("Базис B (1-based) =", [j + 1 for j in B])
    print("A после удаления лишних ограничений:")
    print(np.round(A_red, 6))
    print("b после удаления лишних ограничений:")
    print(np.round(b_red, 6))


Статус: Исходная задача совместна.
Допустимый план x^T = [0. 0. 0.]
Базис B (0-based) = [0]
Базис B (1-based) = [1]
A после удаления лишних ограничений:
[[1. 1. 1.]]
b после удаления лишних ограничений:
[0.]
